In [1]:
import itertools
import os

import h5py
from pressomancy.analysis import H5DataSelector

import numpy as np

from pyanal.global_definitions import A_help_dict

def file_path_to_param(file_path):
    file_name = os.path.basename(file_path).split("-")
    dir_name = os.path.basename(os.path.dirname(file_path)).split("-")
    A=file_name[0][2:]
    DENS=dir_name[1].split("_")[0]
    K=dir_name[1].split("_")[1]
    N_FULL_BOX=dir_name[2].split("_")[0]
    HEIGHT=dir_name[2].split("_")[1]
    H=file_name[2][1:-3]
    seed=file_name[1]
    return (A, DENS, K, HEIGHT, seed, H)

def safe_get_prop(h5_file, prop, str_err):
    try:
        arr = h5_file[f"particles/Elastomer/{prop}/value"][...]
        arr = np.asarray(arr)

        # ensure 3D structure with last axis = components
        if arr.ndim == 1:
            arr = arr[:, None, None]
        elif arr.ndim == 2:
            arr = arr[:, :, None]

        # ensure last dimension is at least 3 (x,y,z)
        if arr.shape[-1] < 3:
            pad_width = [(0, 0)] * arr.ndim
            pad_width[-1] = (0, 3 - arr.shape[-1])
            arr = np.pad(arr, pad_width, mode="constant", constant_values=np.nan)

        return arr

    except Exception as e:
        str_err += f">Failed reading particles/Elastomer/{prop}/value dataset: {e}\n"
        return np.full((1, 1, 3), np.nan)

def safe_read(h5_file, path, str_err):
    try:
        return h5_file[path]
    except Exception as e:
        str_err += f">Failed reading {path} dataset: {e}\n"
    return None

In [ ]:
from repo_paths import DATA_DIR   # repository-relative; set $MAE_DATA_DIR to read the dataset from elsewhere
import os
READ_BASE_DIR = os.path.join(DATA_DIR, "MAE-BoS", "cluster", "sim_data")

N_FULL_BOX = 6000 * 4 # ignore for eger sims

A_list = ["HM", "SM"]
A_CONC = 1.00
DENS_list = [0.20,0.25,0.30]
K_list = ["hard", "soft"]
HEIGHT_list = [10]
H_list = [0,0.2,0.4,0.6,0.8,1,1.5,2,3,4,6,10,15,20,25,30,40,50,60,70]#; H_list=H_list[:-4]
seed_list = [1,2,3,4]

# Times that are expected in each file
target_ts_list = list(range(0, 500+1, 1))

file_missing_list = []
file_corrupted_list = []
file_bad_list = []
file_good_list = []

problematic_count= 0
file_corrupted_count= 0
file_missing_count= 0
global_count= 0
for A, DENS, K, HEIGHT, H, seed in itertools.product(A_list, DENS_list, K_list, HEIGHT_list, H_list, seed_list):
    global_count+= 1

    directory = os.path.join(READ_BASE_DIR, f"{A_help_dict[A]}-{DENS:.2f}_{K}-{N_FULL_BOX}_{HEIGHT}")
    filename = f"cp{A}-{seed}-H{H:.2f}.h5"

    h5_file_path = os.path.join(directory, filename)

    # assert file exists
    if not os.path.isfile(h5_file_path):
        print("-> no file in", h5_file_path)
        file_missing_count+= 1
        file_missing_list.append(h5_file_path)
        continue

    # assert file header is not corrupted
    try:
        with h5py.File(h5_file_path, "r") as h5_file:
            pass
    except Exception as e:
        print(f"***{h5_file_path}***\n    Error:\n        {e}")
        file_corrupted_count+= 1
        file_corrupted_list.append(h5_file_path)
        continue

    # By default store parametrs in bad array, in case of fail
    file_bad_list.append(h5_file_path)

    with h5py.File(h5_file_path, "r") as h5_file:
        str_error = f"-> error messages of {h5_file_path}\n"
        str_error_init_len = len(str_error)
        # Top level keys
        if set(h5_file.keys()) != {'connectivity', 'particles', 'sys', 'sim_inst'}:
            str_error += f">Top level keys not correct:{list(h5_file.keys())}. Expected values ['connectivity', 'particles', 'sys'].\n"
        if set(h5_file["sim_inst"].attrs) != {"kT", "seed"}:
            str_error += f">Simulation instance group attributes not correct:{list(h5_file['sim_inst'].attrs)}. Expected values ['kT', 'seed'].\n"
        if set(h5_file["sys"].attrs) != {"box_l", "periodicity", "time_step"}:
            str_error += f">Sys group system attributes not correct:{list(h5_file['sys'].attrs)}. Expected values ['box_l', 'periodicity', 'time_step'].\n"
        if set(h5_file["connectivity"].keys()) != {'Elastomer'}:
            str_error += f">Connectivity group keys not correct:{list(h5_file['connectivity'].keys())}. Expected values ['Elastomer'].\n"
        if set(h5_file["particles"].keys()) != {'Elastomer'}:
            str_error += f">Particle group keys not correct:{list(h5_file['particles'].keys())}. Expected values ['Elastomer'].\n"
        
        # Connectivity group datasets
        connectivity_set_A_dict = {"HM": {'Elastomer_to_PointDipolePermanent', 'ParticleHandle_to_Elastomer', 'ParticleHandle_to_PointDipolePermanent'},
                                   "SM": {'Elastomer_to_PointDipoleSuperpara', 'ParticleHandle_to_Elastomer', 'ParticleHandle_to_PointDipoleSuperpara'},
                                   "SMfl": {'Elastomer_to_PointDipoleSuperpara', 'ParticleHandle_to_Elastomer', 'ParticleHandle_to_PointDipoleSuperpara'},
                                   "SMdumb": {'Elastomer_to_PointDipoleSuperpara', 'ParticleHandle_to_Elastomer', 'ParticleHandle_to_PointDipoleSuperpara'}}
        if set(h5_file["connectivity/Elastomer"].keys()) != connectivity_set_A_dict[A]:
            str_error += f">Connectivity/Elastomer datasets are not correct:{list(h5_file["connectivity/Elastomer"].keys())}. Expected values {list(connectivity_set_A_dict[A])}.\n"

        # Particles group datasets
        if set(h5_file["particles/Elastomer"].keys()) not in ({'bonds', 'dip', 'director', 'f', 'id', 'image_box', 'pos', 'pos_folded', 'type'}, {'bonds', 'dip', 'director', 'f', 'fix', 'id', 'image_box', 'pos', 'pos_folded', 'type'}):
            str_error += f">Particles/Elastomer datasets are not correct:{list(h5_file['particles/Elastomer'].keys())}. Expected values ['bonds', 'dip', 'director', 'f', 'id', 'image_box', 'pos', 'pos_folded', 'type'].\n"
        # assert id, type, pos and dip for n_parrts
        if h5_file["particles/Elastomer/id/value"].shape[0] < 1:
            str_error+=">No ids saved.\n"
            continue
        else:
            n_part = h5_file["particles/Elastomer/id/value"].shape[1]
            if n_part < 1:
                str_error+=">No ids saved in ts.\n"
        if h5_file["particles/Elastomer/type/value"].shape[1] != n_part:
            str_error+=">Particle types (or ids) are not correctly saved.\n"
        if h5_file["particles/Elastomer/pos/value"].shape[1] != n_part:
            str_error+=">Particle pos (or ids) are not correctly saved.\n"
        if np.min(safe_get_prop(h5_file, "pos", str_error)[:,:,2]) != 0.5:
            str_error+=f">Substrate particle pos are not correctly saved. Particles are not at z={0.5}, instead they are at z={np.min(safe_get_prop(h5_file, "pos", "")[:,:,2])}\n"
        if h5_file["particles/Elastomer/dip/value"].shape[1] != n_part:
            str_error+=">Particle dip (or ids) are not correctly saved.\n"
        if h5_file["particles/Elastomer/pos/value"].shape[2] != 3:
            str_error+=">Particle pos do not have 3 dimenions (x, y, z).\n"
        if h5_file["particles/Elastomer/dip/value"].shape[2] != 3:
            str_error+=">Particle dip do not have 3 dimenions (x, y, z).\n"
        # assert part min and max positions
        tmp = h5_file["particles/Elastomer/pos/value"][:,2].max()
        if tmp < 1.:
            str_error+=f">Particle min z is {tmp}.\n"
        tmp = h5_file["particles/Elastomer/pos/value"][:,2].min()
        if tmp > HEIGHT * 4 - 2:
            str_error+=f">Particle max z is {tmp}.\n"
        # assert bonds
        if h5_file["particles/Elastomer/bonds/value"].shape[0] < 1:
            str_error+=">No bonds saved.\n"
        elif h5_file["particles/Elastomer/bonds/value"].shape[0] > 1:
            str_error+=">WARNING: saved bonds more than 1 time.\n"
        else:
            # if len([x for part_bonds in h5_file["particles/Elastomer/bonds/value"][0,:] for bonds in part_bonds for x in bonds]) == 0:
            #     str_error+=">All bonds saved are empty.\n"
            pass
        # assert time
        ts_list = list(map(int, h5_file["particles/Elastomer/id/time"]))
        max_ts_list = max(ts_list); max_target_ts_list = max(target_ts_list)
        if max_ts_list < max_target_ts_list:
            str_error+=f">Simulation did not reach desired time: max saved time {max_ts_list}; target {max_target_ts_list}.\n"
        elif ts_list != sorted(ts_list):
            str_error+=f">Times were not saved in order. This is a problem: ts_list {ts_list}; target {target_ts_list}.\n"
        elif not set(target_ts_list).issubset(set(ts_list)):
            str_error+=f">Times were not saved correctly. This is a problem: ts_list {ts_list}; target {target_ts_list}.\n"
        
    if len(str_error) < str_error_init_len:
        raise ValueError(f"What the fuck não abriu o ficheiro e não deu erro ou apagou a mensagem ou que: {len(str_error)} < {str_error_init_len}")
    elif len(str_error) > str_error_init_len:
        print(str_error)
        problematic_count+= 1
        continue

    # if pased all tests - remove form bad lit, and append to good list
    file_bad_list = file_bad_list[:-1]
    file_good_list.append(h5_file_path)

assert len(file_missing_list) == file_missing_count
assert len(file_corrupted_list) == file_corrupted_count
assert len(file_bad_list) == problematic_count
assert len(file_good_list) == global_count - (file_missing_count + file_corrupted_count + problematic_count)

def list_to_formated_str(lista):
    str_out="list["
    for x in lista:
        str_out += "\n" + str(x) + ","
    str_out += "\n]"
    return str_out

print(f"\nMissing list:\n{list_to_formated_str(file_missing_list)}")
print(f"\nCorrupted list:\n{list_to_formated_str(file_corrupted_list)}")
print(f"\nBad list:\n{list_to_formated_str(file_bad_list)}")
print(f"\nGood list:\n{list_to_formated_str(file_good_list)}")

print(f"\nMising params:\n{list_to_formated_str(list(map(file_path_to_param, file_missing_list)))}")
print(f"\nCorrupted params:\n{list_to_formated_str(list(map(file_path_to_param, file_corrupted_list)))}")
print(f"\nBad params:\n{list_to_formated_str(list(map(file_path_to_param, file_bad_list)))}")
print(f"\nGood params:\n{list_to_formated_str(list(map(file_path_to_param, file_good_list)))}")

print(f"\nAnalyzed {global_count} combinations of parameters.\n")
print(f"Files missing: {file_missing_count}/{global_count}")
print(f"Files corrupted: {file_corrupted_count}/{(global_count-file_missing_count)}")
print(f"Problematic files: {problematic_count}/{global_count-file_missing_count-file_corrupted_count}")
print(f"\nTotal number of correctly saved data: {global_count-file_missing_count-file_corrupted_count-problematic_count}/{global_count}")

***$DATA/MAE-BoS/cluster/sim_data/pdp-0.30_hard-24000_10/cpHM-1-H0.40.h5***
    Error:
        Unable to synchronously open file (bad object header version number)
***$DATA/MAE-BoS/cluster/sim_data/pdp-0.30_hard-24000_10/cpHM-1-H0.80.h5***
    Error:
        Unable to synchronously open file (bad object header version number)
-> error messages of $DATA/MAE-BoS/cluster/sim_data/pdp-0.30_hard-24000_10/cpHM-1-H1.50.h5
>Simulation did not reach desired time: max saved time 368; target 500.

-> error messages of $DATA/MAE-BoS/cluster/sim_data/pdp-0.30_hard-24000_10/cpHM-1-H3.00.h5
>Simulation did not reach desired time: max saved time 373; target 500.

-> error messages of $DATA/MAE-BoS/cluster/sim_data/pdp-0.30_hard-24000_10/cpHM-1-H70.00.h5
>Simulation did not reach desired time: max saved time 499; target 500.

-> error messages of $DATA/MAE-BoS/cluster/sim_data/pdp-0.30_soft-24000_10/cpHM-1-H0.20.h5
>Simulation did not reach desired time: max saved time 366; target 500.

-> error messag

In [ ]:
READ_BASE_DIR = os.path.join(DATA_DIR, "MAE-BoS", "cluster", "sim_data-h5")

N_FULL_BOX = 6000 * 4 # ignore for eger sims

A_list = ["HM", "SM"]
A_CONC = 1.00
DENS_list = [0.30]
K_list = ["hard"]
HEIGHT_list = [10]
seed_list = [1]

H_list  = [0  , 0.2, 0.4, 0.6, 0.8, 1  , 1.5, 2, 3, 4, 6 , 3]
H0_list = [0.2, 0.4, 0.6, 0.8, 1  , 1.5, 2  , 3, 4, 6, 10, 1.5]

# Times that are expected in each file
target_ts_list = [500] #list(range(100, 500+1, 100))

file_bad_list = []
file_good_list = []

def file_path_to_param(file_path):
    file_name = os.path.basename(file_path).split("-")
    dir_name = os.path.basename(os.path.dirname(file_path)).split("-")
    A=file_name[0][2:]
    DENS=dir_name[2].split("_")[0]
    K=dir_name[2].split("_")[1]
    N_FULL_BOX=dir_name[3].split("_")[0]
    HEIGHT=dir_name[3].split("_")[1]
    H=file_name[2][1:]
    seed=file_name[1]
    return (A, DENS, K, HEIGHT, seed, H)

problematic_count= 0
file_corrupted_count= 0
file_missing_count= 0
global_count= 0
for A, DENS, K, HEIGHT, (H, H0), seed in itertools.product(A_list, DENS_list, K_list, HEIGHT_list, zip(H_list, H0_list), seed_list):
    global_count+= 1

    directory = os.path.join(READ_BASE_DIR, f"{A_help_dict[A]}-{A_CONC:.2f}-{DENS:.2f}_{K}-{N_FULL_BOX}_{HEIGHT}-reverse")
    filename = f"cp{A}-{seed}-H{H:.2f}-fromH{H0:.2f}.h5"

    h5_file_path = os.path.join(directory, filename)

    # By default store parametrs in bad array, in case of fail
    file_bad_list.append(h5_file_path)

    # assert file exists
    if not os.path.isfile(h5_file_path):
        print("-> no file in", h5_file_path)
        file_missing_count+= 1
        continue

    # assert file is not corrupted
    try:
        with h5py.File(h5_file_path, "r") as h5_file:
            pass
    except Exception as e:
        print(f"***{h5_file_path}***\n    Error:\n        {e}")
        file_corrupted_count+= 1
        continue

    with h5py.File(h5_file_path, "r") as h5_file:
        str_error = f"-> error messages of {h5_file_path}\n"
        str_error_init_len = len(str_error)
        # Top level keys
        if set(h5_file.keys()) != {'connectivity', 'particles', 'sys'}:
            str_error += f">Top level keys not correct:{list(h5_file.keys())}. Expected values ['connectivity', 'particles', 'sys'].\n"
        if set(h5_file["sys"].attrs) != {"box_l", "periodicity", "time_step"}:
            str_error += f">Sys group system attributes not correct:{list(h5_file["sys"].attrs)}. Expected values ['box_l', 'periodicity', 'time_step'].\n"
        if set(h5_file["connectivity"].keys()) != {'Elastomer'}:
            str_error += f">Connectivity group keys not correct:{list(h5_file["connectivity"].keys())}. Expected values ['Elastomer'].\n"
        if set(h5_file["particles"].keys()) != {'Elastomer'}:
            str_error += f">Particle group keys not correct:{list(h5_file["particles"].keys())}. Expected values ['Elastomer'].\n"
        
        # Connectivity group datasets
        connectivity_set_A_dict = {"HM": {'Elastomer_to_PointDipolePermanent', 'ParticleHandle_to_Elastomer', 'ParticleHandle_to_PointDipolePermanent'},
                                   "SM": {'Elastomer_to_PointDipoleSuperpara', 'ParticleHandle_to_Elastomer', 'ParticleHandle_to_PointDipoleSuperpara'},
                                   "SMfl": {'Elastomer_to_PointDipoleSuperpara', 'ParticleHandle_to_Elastomer', 'ParticleHandle_to_PointDipoleSuperpara'}}
        if set(h5_file["connectivity/Elastomer"].keys()) != connectivity_set_A_dict[A]:
            str_error += f">Connectivity/Elastomer datasets are not correct:{list(h5_file["connectivity/Elastomer"].keys())}. Expected values {list(connectivity_set_A_dict[A])}.\n"

        # Particles group datasets
        if set(h5_file["particles/Elastomer"].keys()) != {'bonds', 'dip', 'director', 'f', 'id', 'image_box', 'pos', 'pos_folded', 'type'}:
            str_error += f">Particles/Elastomer datasets are not correct:{list(h5_file["particles/Elastomer"].keys())}. Expected values ['bonds', 'dip', 'director', 'f', 'id', 'image_box', 'pos', 'pos_folded', 'type'].\n"
        # assert id, type, pos and dip for n_parrts
        if h5_file["particles/Elastomer/id/value"].shape[0] < 1:
            str_error+=">No ids saved.\n"
        else:
            n_part = h5_file["particles/Elastomer/id/value"].shape[1]
            if n_part < 1:
                str_error+=">No ids saved in ts.\n"
        if h5_file["particles/Elastomer/type/value"].shape[1] != n_part:
            str_error+=">Particle types (or ids) are not correctly saved.\n"
        if h5_file["particles/Elastomer/pos/value"].shape[1] != n_part:
            str_error+=">Particle pos (or ids) are not correctly saved.\n"
        if h5_file["particles/Elastomer/dip/value"].shape[1] != n_part:
            str_error+=">Particle dip (or ids) are not correctly saved.\n"
        if h5_file["particles/Elastomer/pos/value"].shape[2] != 3:
            str_error+=">Particle pos do not have 3 dimenions (x, y, z).\n"
        if h5_file["particles/Elastomer/dip/value"].shape[2] != 3:
            str_error+=">Particle dip do not have 3 dimenions (x, y, z).\n"
        # assert bonds
        if h5_file["particles/Elastomer/bonds/value"].shape[0] < 1:
            str_error+=">No bonds saved.\n"
        elif h5_file["particles/Elastomer/bonds/value"].shape[0] > 1:
            str_error+=">WARNING: saved bonds more than 1 time.\n"
        else:
            bond_list_single_ts=[]
            for pt in range(n_part):
                try:
                    assert len(h5_file["particles/Elastomer/bonds/value"][0, pt]) > 0
                    bond_list_single_ts.append(h5_file["particles/Elastomer/bonds/value"][0, pt])
                except:
                    bond_list_single_ts.append([])
            if len(bond_list_single_ts) == 0:
                str_error+=">All bonds saved are empty.\n"
        # assert time
        ts_list = list(map(int, h5_file["particles/Elastomer/id/time"]))
        max_ts_list = max(ts_list); max_target_ts_list = max(target_ts_list)
        if max_ts_list < max_target_ts_list:
            str_error+=f">Simulation did not reach desired time: max saved time {max_ts_list}; target {max_target_ts_list}.\n"
        elif ts_list != sorted(ts_list):
            str_error+=f">Times were not saved in order. This is a problem: ts_list {ts_list}; target {target_ts_list}.\n"
        elif not set(target_ts_list).issubset(set(ts_list)):
            str_error+=f">Times were not saved correctly. This is a problem: ts_list {ts_list}; target {target_ts_list}.\n"
        
    if len(str_error) < str_error_init_len:
        raise ValueError(f"What the fuck não abriu o ficheiro e não deu erro ou apagou a mensagem ou que: {len(str_error)} < {str_error_init_len}")
    elif len(str_error) > str_error_init_len:
        print(str_error)
        problematic_count+= 1
        continue

    # if pased all tests - remove form bad lit, and append to good list
    file_bad_list = file_bad_list[:-1]
    file_good_list.append(h5_file_path)

assert len(file_bad_list) == (file_missing_count + file_corrupted_count + problematic_count)
assert len(file_good_list) == global_count - (file_missing_count + file_corrupted_count + problematic_count)

print(f"\nBad list:\n{file_bad_list}")
print(f"\nGood list:\n{file_good_list}")

print(f"\nBad params:\n{list(map(file_path_to_param, file_bad_list))}")
print(f"\nGood params:\n{list(map(file_path_to_param, file_good_list))}")

print(f"\nAnalyzed {global_count} combinations of parameters.\n")
print(f"Files missing: {file_missing_count}/{global_count}")
print(f"Files corrupted: {file_corrupted_count}/{(global_count-file_missing_count)}")
print(f"Problematic files: {problematic_count}/{global_count-file_missing_count-file_corrupted_count}")
print(f"\nTotal number of correctly saved data: {global_count-file_missing_count-file_corrupted_count-problematic_count}/{global_count}")